In [1]:
path = "data_base/bbc_text_cls_adapted.csv"

In [2]:
import pandas as pd
import numpy as np
import random
import math

In [3]:
data = pd.read_csv(filepath_or_buffer = path)

In [4]:
labels = data["labels"].unique()

In [5]:
X = data["text"]
y = data["labels"]

In [6]:
if y.dtype == object or str:
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)
    y = pd.DataFrame(y)

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.5, random_state = 1, stratify = y)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')

X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed = vectorizer.transform(X_test)

In [10]:
# X_train_transformed = X_train_transformed.toarray()
# X_test_transformed = X_test_transformed.toarray()

In [11]:
indices_treino = y_train.index
columns = data.iloc[indices_treino, 2:]

In [12]:
matrix = np.array(columns)
print(matrix)

[['tech' 'entertainment' 'entertainment' ... 'tech' 'sport' 'business']
 ['sport' 'sport' 'sport' ... 'business' 'tech' 'tech']
 ['sport' 'sport' 'sport' ... 'sport' 'sport' 'sport']
 ...
 ['business' 'business' 'business' ... 'sport' 'tech' 'tech']
 ['sport' 'sport' 'politics' ... 'business' 'tech' 'tech']
 ['entertainment' 'entertainment' 'entertainment' ... 'business' 'sport'
  'sport']]


In [13]:
list_3 = matrix.tolist()

In [14]:
#list_ =  [list(set(l)) for l in matrix]

In [15]:
#list_2 = [i for i in list_ if len(i)>1]

In [16]:
len_ = len(list_3)

In [17]:
#individual = np.random.randint(0,data["labels"].nunique(),len_)

In [18]:
individual = random.choices(labels, k=len_)

In [19]:
from sklearn.naive_bayes import MultinomialNB

In [20]:
naive_bayes_mult = MultinomialNB()

In [21]:
from sklearn.metrics import r2_score

In [22]:
def removal_function(individual,n_remove):
    individual = list(individual)
    length = len(individual)
    indexes = random.sample(range(0,length),n_remove)
    # values_list = []
    # for _ in indexes:
    #     values_list.append(individual[_])

    return(indexes)

In [23]:
# def determinator(tiny_list):
#     weights =[]
#     for v in tiny_list:
#         weights.append(dictionary_.get(v))
#     choice = random.choices(tiny_list, weights=weights, k=1)
#     return(choice)

In [24]:
# def insertion_function(values_list,indexes,individual,priors_array,labels):
#     list_2 = global list_2 
#     values_transformados = [determinator(list_2[v]) for v in indexes]
#     # f = lambda x: 1-x
#     # values_transformed = [f(x) for x in values_list]
#     for index, _ in enumerate(indexes):
#         individual[_] = values_transformed[index]
        
#     return(individual,values_transformed)
    

In [25]:
def determinator(tiny_list):
    choice = random.choice(tiny_list)
    return(choice)

In [26]:
def insertion_function(indexes,individual,list_3): 
    individual = list(individual)
    valores_transformados = [determinator(list_3[v]) for v in indexes]
    for index, _ in enumerate(indexes):
        individual[_] = valores_transformados[index]
        
    return(individual)
    

In [27]:
def make_list_proba(df_):
    total_proba_ = df_[0].prod()
    total_proba_2 = df_[1].prod()
    total_proba_3 = df_[2].prod()
    total_proba_4 = df_[3].prod()
    total_proba_5 = df_[4].prod()
    list_proba_ = []
    list_proba_.append(total_proba_)
    list_proba_.append(total_proba_2)
    list_proba_.append(total_proba_3)
    list_proba_.append(total_proba_4)
    list_proba_.append(total_proba_5)
    return(list_proba) 

In [28]:
def fitness_function(individual,list_3,X_train_transformed):
    list_ = list_3.copy()
    list_index = []
    df_ = pd.DataFrame()
    cont = 0
    for index,values in enumerate(list_):
        if len(values) > 1:
            list_[index] = individual[cont]
            cont+=1

    list_ = [item[0] if isinstance(item, list) else item for item in list_]

    naive_bayes_mult.fit(X_train_transformed,list_)
    naive_bayes_mult.predict(X_train_transformed)
    df_ = naive_bayes_mult.predict_log_proba(X_train_transformed)
    list_before_sum = [max(i) for i in df_]
    fitness = sum(list_before_sum)
    
    return(fitness)

In [29]:
def implementation_function(individual,n_remove,labels,X_train_transformed,list_3,k):
    best_fitness = float('-inf')
    best_individual = []
    fitness = 0
    euler = math.e
    
    for _ in range(k):
        indexes = removal_function(individual,n_remove)
        individual = insertion_function(indexes,individual,list_3)
        fitness = fitness_function(individual,list_3,X_train_transformed)
        # print(fitness)
        # print(best_fitness)
        
        if fitness > best_fitness:
            best_individual = individual
            best_fitness = fitness
        else:
            e_dif = euler ** (best_fitness-fitness)
            if e_dif > round(random.uniform(1,e_dif*2 ),2): # Ex: 54.32 random.randint(0,(e_dif*2)):
                best_individual = individual
                best_fitness = fitness

    return(best_fitness,best_individual)

In [30]:
best_fitness,best_individual = implementation_function(individual,3,labels,X_train_transformed,list_3,80000)

In [31]:
print(best_fitness)
print(best_individual)

-961.3786306538233
['politics', 'sport', 'entertainment', 'entertainment', 'entertainment', 'entertainment', 'politics', 'sport', 'sport', 'sport', 'business', 'entertainment', 'entertainment', 'entertainment', 'sport', 'sport', 'tech', 'politics', 'entertainment', 'sport', 'politics', 'politics', 'politics', 'sport', 'business', 'business', 'sport', 'politics', 'business', 'sport', 'tech', 'business', 'tech', 'politics', 'business', 'sport', 'sport', 'tech', 'business', 'entertainment', 'tech', 'business', 'entertainment', 'tech', 'entertainment', 'sport', 'entertainment', 'business', 'politics', 'entertainment', 'politics', 'politics', 'sport', 'tech', 'politics', 'politics', 'business', 'tech', 'tech', 'sport', 'business', 'business', 'business', 'sport', 'entertainment', 'entertainment', 'business', 'business', 'sport', 'politics', 'sport', 'sport', 'politics', 'entertainment', 'tech', 'business', 'politics', 'tech', 'politics', 'business', 'business', 'sport', 'sport', 'business',

In [32]:
print("hi")

hi
